#Assignment 2 – Knowledge Graph

Perform Named-Entity Recognition (NER) on the corpus documents

Identify, and list, the en es and their labels that are recognized by spaCy’s default
medium-sized model (‘en_core_web_md’);

In [1]:
from src.Project2.ner_baseline import run_baseline_ner, summarize, save_entity_frequencies_to_csv
df = run_baseline_ner("../../data/train")
save_entity_frequencies_to_csv(df, "shakespeare_entities.csv")
summarize(df)

Looking in: E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\data\train
Matched files:
 - Shakespeare_Macbeth.txt
 - Shakespeare_Midsummer_Nights_Dream.txt
 - Shakespeare_Much_Ado_About_Nothing.txt
 - Shakespeare_Romeo_and_Juliet.txt
Loaded 4 text files
Processing: Shakespeare_Macbeth.txt
Processing: Shakespeare_Midsummer_Nights_Dream.txt
Processing: Shakespeare_Much_Ado_About_Nothing.txt
Processing: Shakespeare_Romeo_and_Juliet.txt
Saved entity frequencies to: shakespeare_entities.csv

=== Label Distribution ===
label
PERSON         2091
ORG             753
GPE             366
CARDINAL        250
DATE            230
TIME            198
ORDINAL         119
NORP             64
WORK_OF_ART      48
PRODUCT          44
LOC              17
LANGUAGE          9
QUANTITY          8
MONEY             5
EVENT             5
FAC               4
LAW               1
Name: count, dtype: int64

=== Entities with Frequency ===
     

2. Identify and list, any en es that have been mislabeled by the default model;
3. Identify any en es that are missing labels, or do not have a default label to describe
their intended meaning;

In [2]:
from src.Project2.ner_mislabeled_finder import load_entity_csv, build_expected_entities, find_mislabeled_entities, find_missing_entities, find_no_good_default_label_entities

freq_df = load_entity_csv("shakespeare_entities.csv")
expected_entities = build_expected_entities()

mislabeled_df = find_mislabeled_entities(freq_df, expected_entities)
missing_df = find_missing_entities(freq_df, expected_entities)
no_default_df = find_no_good_default_label_entities(freq_df)

print("\n=== Step 2: Mislabeled Entities ===")
print(mislabeled_df.to_string(index=False))

print("\n=== Step 3A: Missing Entities ===")
print(missing_df.to_string(index=False))

print("\n=== Step 3B: No Good Default Label ===")
print(no_default_df.to_string(index=False))


=== Step 2: Mislabeled Entities ===
   entity predicted_label expected_label  count
  CLAUDIO             ORG         PERSON    104
    Romeo             ORG         PERSON     74
    Paris             GPE         PERSON     34
     Hero             ORG         PERSON     30
   Helena             ORG         PERSON     24
Demetrius             ORG         PERSON     23
    Cupid             ORG         PERSON     20
 Beatrice             ORG         PERSON     17
 Montague             GPE         PERSON     15
   Hermia             GPE         PERSON     15
     HERO             ORG         PERSON     14
  THESEUS             ORG         PERSON     11
  Leonato             GPE         PERSON      9
  Titania         PRODUCT         PERSON      7
   Verona          PERSON            GPE      7
  Pyramus             ORG         PERSON      7
      DON             ORG         PERSON      6
  Antonio             GPE         PERSON      6
  Titania             ORG         PERSON      6
   

4. For all en es iden fied in step 3 above, fine-tune the model to correct any
- Mislabeled entities ,
- Missing-label entities ,
- Pronoun resolution, coreference resolution – go beyond the ‘1-sentence-back’ model to write your own custom code for pronoun resolution.

5. List out all the entities with their appropriate labels. Use a spreadsheet/table to include this informa on in your report

In [3]:
from src.Project2.ner_finetuning import load_corpus_text, build_training_examples_from_text, build_training_examples_from_corpus, combine_training_examples, TRAIN_DATA,  resolve_pronouns_custom, fine_tune_shakespeare_ner, export_final_entity_table

data_dir = '../../data/train'
corpus_text = load_corpus_text(data_dir)

weak_examples_text = build_training_examples_from_text(corpus_text, expected_entities, window_size=300, max_examples_per_entity=10)
len(weak_examples_text), weak_examples_text[:3]

(3000, 1006)

In [ ]:
weak_examples_sent = build_training_examples_from_corpus(corpus_text, expected_entities, min_entities_per_example=1, max_examples=3000)
training_data = combine_training_examples(TRAIN_DATA, weak_examples_sent)
len(weak_examples_sent), len(training_data)

In [4]:
nlp_ft = fine_tune_shakespeare_ner(
    train_data=training_data,
    output_dir='shakespeare_ner_model',
    base_model='en_core_web_md',
    n_iter=40,
    entity_labels=expected_entities,
)

sample_text = corpus_text[:10000]
doc = nlp_ft(sample_text)
coref_df = resolve_pronouns_custom(doc)
coref_df.head(30)

C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Now, good sweet Nurse,—O Lord, why look’st thou sa..." with entities "[(16, 21, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Speak, Pyramus.—Thisbe, stand forth." with entities "[(7, 14, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities c

Iteration 1/20 - Losses: {'ner': 1685.3299407177692}
Iteration 2/20 - Losses: {'ner': 1160.2082923735998}
Iteration 3/20 - Losses: {'ner': 984.0244489713456}
Iteration 4/20 - Losses: {'ner': 836.4153736407044}
Iteration 5/20 - Losses: {'ner': 776.5605291030868}
Iteration 6/20 - Losses: {'ner': 606.998570158244}
Iteration 7/20 - Losses: {'ner': 558.0560169416656}
Iteration 8/20 - Losses: {'ner': 530.0368708319521}
Iteration 9/20 - Losses: {'ner': 485.9877083021155}
Iteration 10/20 - Losses: {'ner': 385.17661878621533}
Iteration 11/20 - Losses: {'ner': 367.3390066469395}
Iteration 12/20 - Losses: {'ner': 346.4954393347143}
Iteration 13/20 - Losses: {'ner': 292.5587860177532}
Iteration 14/20 - Losses: {'ner': 268.92969737183273}
Iteration 15/20 - Losses: {'ner': 247.46321400270102}
Iteration 16/20 - Losses: {'ner': 230.56479004904506}
Iteration 17/20 - Losses: {'ner': 217.53926599791612}
Iteration 18/20 - Losses: {'ner': 182.664024449268}
Iteration 19/20 - Losses: {'ner': 174.847839183122

,pronoun,sentence,resolved_to,rationale
0,He,"He can report, As seemeth by his plight, of th...",DUNCAN,Most recent male-compatible entity in rolling ...
1,his,"He can report, As seemeth by his plight, of th...",DUNCAN,Most recent male-compatible entity in rolling ...
2,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
3,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
4,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
5,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
6,him,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
7,he,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
8,him,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...
9,his,But all’s too weak; For brave Macbeth (well he...,Macbeth,Most recent male-compatible entity in rolling ...


In [5]:
final_entities_df = export_final_entity_table(
    text=corpus_text,
    model_path='shakespeare_ner_model',
    output_csv='final_entities.csv'
)
final_entities_df.head(50)

Saved final entity table to E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\src\Project2\final_entities.csv


,entity,label,frequency
0,Athens,GPE,20
1,Aurora,GPE,1
2,Betroth’d,GPE,1
3,Birnam,GPE,10
4,Crete,GPE,2
5,Egypt,GPE,1
6,England,GPE,8
7,Fife,GPE,4
8,Glamis,GPE,8
9,Hobgoblin,GPE,1
